# 24 — First-Time Auto-Bootstrap: image → spec

`bootstrap.estimate_initial_spec(image, wavelength, pxY, ...)` does the
minimum-input handshake: a raw image and a wavelength, no prior
geometry knowledge. Internally:

1. `estimate_BC_from_image` finds the beam centre from ring symmetry.
2. `ring_detect.detect_rings` locates Bragg rings in the radial profile.
3. `ring_detect.suggest_material` matches the ring positions against
   the built-in `CALIBRANTS` database (CeO₂, LaB₆, Si, Al₂O₃).
4. Combines those with the wavelength to back out an initial Lsd.

You get a fully populated `IntegrationSpec` ready for refinement
(`midas-calibrate-v2.calibrate(...)`).

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2 import estimate_BC_from_image, estimate_initial_spec
from midas_integrate_v2.ring_detect import detect_rings, suggest_material, CALIBRANTS

# Synthetic CeO2 image (you'd load a real one).
NY = NZ = 1024
y, z = np.meshgrid(np.arange(NY), np.arange(NZ))
R = np.sqrt((y - 512)**2 + (z - 512)**2)
img = np.zeros((NZ, NY), dtype=np.float32)
for r0 in (110, 130, 175, 210, 230, 270):
    img += 1000.0 * np.exp(-0.5 * ((R - r0)/1.0)**2)

# Step 1 — BC alone
BC = estimate_BC_from_image(img)
print(f'BC = {BC}')

# Step 2 — full spec bootstrap
spec = estimate_initial_spec(
    img, wavelength_A=0.184139, pxY=200.0,
    NrPixelsY=NY, NrPixelsZ=NZ,
    calibrant='CeO2',         # or None to auto-detect via suggest_material
)
print(f'Lsd = {float(spec.Lsd)/1000:.3f} mm')
print(f'BC  = ({float(spec.BC_y):.2f}, {float(spec.BC_z):.2f})')

## Auto material-ID

`suggest_material(detected_rings, wavelength)` ranks the calibrants by
goodness-of-fit. Useful when you forgot to label your file or the
beamtime mixed samples.

In [ ]:
print('Known calibrants:', list(CALIBRANTS.keys()))